# Aralin 10 - Mga Ahente ng AI sa Produksyon

Sa araling ito matututuhan mo ang **mga pattern sa produksyon** para sa mga ahente ng AI gamit ang Microsoft Agent Framework (Python). Tinatalakay namin:

- **Pagsubaybay** — pagdaragdag ng timing at logging sa mga interaksyon ng ahente
- **Ebalwasyon** — paggamit ng evaluator na ahente para i-skor ang kalidad ng tugon
- **Pamamahala ng Gastos** — mga estratehiya para sa pag-optimize ng token at pagpili ng modelo

Ang senaryo ay isang **ahente ng paglalakbay** na tumutulong sa mga gumagamit na magplano ng mga biyahe, na may karagdagang pagmamanman at ebalwasyon.


## Pagsasaayos


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Mga Pagsasaalang-alang sa Produksyon

Ang paglilipat ng mga AI agent mula sa mga prototype patungo sa produksyon ay nangangailangan ng maingat na pagtuon sa tatlong haligi:

1. **Observability** — Kailangan mong magkaroon ng kakayahang makita kung ano ang ginagawa ng agent, gaano katagal ito tumatagal, at aling mga tool ang tinatawag nito. Kung walang pagsusubaybay at pagtatala, halos imposible ang pag-debug ng mga isyu sa produksyon.

2. **Evaluation** — Ang awtomatikong mga pagsusuri ng kalidad ay tinitiyak na ang mga tugon ng agent ay nananatiling tumpak, kumpleto, at kapaki-pakinabang sa paglipas ng panahon. Ang isang tagasuri na agent ay maaaring magbigay ng iskor sa mga tugon laban sa itinakdang mga pamantayan.

3. **Cost Management** — Direktang nakakaapekto sa gastos ang paggamit ng mga token. Ang mga estratehiya tulad ng pag-optimize ng prompt, pagpili ng modelo, at paggamit ng cache ay tumutulong panatilihing kontrolado ang mga gastos nang hindi isinusuko ang kalidad.


## Pagbuo ng isang Observable na Ahente

Tinutukoy namin ang mga tool para sa paglalakbay at binabalutan ang pagtawag sa ahente ng timing upang masubaybayan ang latency. Sa production, i-integrate mo ito sa OpenTelemetry o sa isang katulad na tracing backend.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Mga Pattern ng Pagsusuri

Isang karaniwang pattern sa produksiyon ay ang gumamit ng pangalawang ahente bilang isang **tagasuri**. Sinusuri ng tagasuri ang tugon ng pangunahing ahente laban sa mga paunang itinakdang pamantayan tulad ng pagiging kumpleto, katumpakan, at pagiging kapaki-pakinabang.

Pinapahintulutan nito:
- Awtomatikong mga gate ng kalidad bago maabot ng mga tugon ang mga gumagamit
- Pagtuklas ng regression kapag nagbago ang mga prompt o mga modelo
- Patuloy na pagmamanman ng pagganap ng ahente sa paglipas ng panahon


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Mga Estratehiya sa Pamamahala ng Gastos

Mahalaga ang pagkontrol sa gastos para sa mga AI agent na nasa produksiyon. Narito ang mga pangunahing estratehiya:

| Estratehiya | Paglalarawan |
|---|---|
| **Pag-optimize ng prompt** | Panatilihing maigsi ang mga instruksiyon ng sistema. Alisin ang mga paulit-ulit na konteksto upang mabawasan ang mga input token. |
| **Pagpili ng modelo** | Gumamit ng mas maliliit at mas murang mga modelo (hal. GPT-4o-mini) para sa mga simpleng gawain tulad ng klasipikasyon o pagkuha, at ireserba ang mas malalaking modelo para sa mas kumplikadong pangangatwiran. |
| **Pag-cache** | I-cache ang mga resulta ng tool at ang mga madalas na query upang maiwasan ang mga paulit-ulit na tawag sa API. |
| **Badyet ng token** | Magtakda ng mga limitasyon na `max_tokens` upang maiwasan ang hindi inaasahang mahahabang tugon. |
| **Pagsasama-sama** | Pagsamahin ang maramihang mga query ng user sa isang API call kung maaari. |

Sa praktika, epektibo ang isang naka-tier na pamamaraan: ipadala ang mga payak na kahilingan sa isang mabilis at murang modelo, at ilipat lamang ang mga kumplikadong query sa isang mas kapable na modelo.


## Buod

Sa araling ito natutunan mo kung paano:

1. **Magdagdag ng obserbabilidad** sa mga interaksyon ng agent gamit ang pagtatala ng oras at pag-log, na naglalatag ng pundasyon para sa pagsubaybay at pagmamanman.
2. **Suriin ang mga tugon ng agent** nang awtomatiko gamit ang isang evaluator agent na nagbibigay ng iskor sa kabuuan, kawastuhan, at kapaki-pakinabang.
3. **Pamahalaan ang mga gastos** sa pamamagitan ng pag-optimize ng prompt, pagpili ng modelo, caching, at mga badyet ng token.

Tinutulungan ng mga pattern na ito sa produksyon na matiyak na ang iyong mga AI agent ay maaasahan, nasusukat, at matipid sa gastos sa malakihang paggamit.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Paunawa:
Ang dokumentong ito ay isinalin gamit ang serbisyong AI na pagsasalin na Co-op Translator (https://github.com/Azure/co-op-translator). Bagaman nagsusumikap kami para sa katumpakan, pakitandaan na ang mga awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o kamalian. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na awtoritatibong sanggunian. Para sa mga kritikal na impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang hindi pagkakaunawaan o maling interpretasyon na nagmumula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
